**Fin 585**  
**Diether**  
**Problem Set**  
**Cross-Sectional Tests of the CAPM**  

**Overview**  

In this problem set you test the CAPM using a cross-sectional framework. Specifically, you test whether the CAPM holds using the Fama-MacBeth (1973) methodology. Before proceeding you need the datafile for this assignment. It's on Learning Suite ('mstk_fm_29-63.csv') or you can download it directly via this link: [Month Stock file: 29-63](http://diether.org/prephd/11-mstk_fm_29-63.csv). The data are a panel with returns from February 1929 to June of 1963 for all stocks common stocks on CRSP during the period:

| Variable | Description                                                               |
|----------|---------------------------------------------------------------------------|
| permno   | stock identifier                                                          |
| caldt    | the calendar month                                                        |
| ret      | return (from the close of the end of month t − 1 to the close of month t) |
| beta     | the estimated beta, estimated using data from months t − 60 to t − 1      |
| melag    | market-cap lagged one month                                               |
| bmlag    | book to market equity lagged as in Fama-French (1992)                     |

Essentially, your overall task for this homework is an out of sample test of Fama and French (1992).

For questions that require some write-up, create a markdown cell (use the Cell Toolbar)  and write your answer in the markdown cell (this cell is a markdown cell and here is a [markdown cheat sheet](https://github.com/adam-p/markdown-here/wiki/Markdown-Cheatsheet)). <br><br>


**Tasks and Questions**  

1. I want you to test the CAPM by estimating Fama-MacBeth regressions of the following form: 
\begin{align*}
r_{it} &= \gamma_{0t} + \gamma_{1t}\hat{\beta}_{it} + \gamma_{2t}log(ME_{i,t-1})
                      + \gamma_{3t}log([\tfrac{B}{M}]_{i,t-1}) + \nu_{it}
\end{align*}
Explain how estimating these Fama-MacBeth regressions is a test of the CAPM. `Pandas` does *not* have a built in Fama-MacBeth function. However, the Fin 585 Library does have a Fama MacBeth regression function: [Fama MacBeth Docs](https://fin-library.readthedocs.io/en/latest/fama_macbeth.html). Use it to estimate the regression above.

2. Based on your results in question (2), can you reject the CAPM? Explain. 

3. This time I want you to estimate Fama-MacBeth regressions of the following form: 
\begin{align*}
r_{it} = \gamma_{0t} + \gamma_{1t}\hat{\beta}_{it} + \nu_{it}
\end{align*}
Report the results of your Fama-MacBeth regressions in a table (it should include standard errors and t-statistics).

4. Based on your results in question (4), can you reject the CAPM? Explain. Is it even possible to ever reject the CAPM with a regression specification like the one you used in question (4)?

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from finance_byu.fama_macbeth import fama_macbeth_master as fmFunc
from finance_byu.fama_macbeth import fm_summary

In [7]:
df = pd.read_csv('11-mstk_fm_29-63.csv',parse_dates=['caldt'])

# prepping dataset:
df['ln_melag'] = np.log(df.melag)
df['ln_bmlag'] = np.log(df.bmlag)

df.head()

,permno,caldt,ret,beta,melag,bmlag,ln_melag,ln_bmlag
0,10006,1929-02-28,0.002519,0.63048,59.55,1.077,4.086816,0.074179
1,10006,1929-03-28,0.027638,0.62575,59.70,1.077,4.089332,0.074179
2,10006,1929-04-30,-0.022333,0.60519,60.45,1.077,4.101817,0.074179
3,10006,1929-05-31,-0.045685,0.60637,59.10,1.077,4.079231,0.074179
4,10006,1929-06-29,0.042553,0.60761,56.40,1.077,4.032469,0.074179


In [ ]:
def ols_coef(x,formula):
    return smf.ols(formula,data=x).fit().params

# adding ln_melag and ln_bmlag to the given code for the regression.
lm = 'ret ~ 1 + beta + ln_melag + ln_bmlag'
gamma = df.groupby('caldt').apply(ols_coef,lm,include_groups=False)
print(gamma)

            Intercept      beta  ln_melag  ln_bmlag
caldt                                              
1929-01-31   0.023166 -0.008054  0.005281 -0.003997
1929-02-28  -0.003124  0.014962 -0.000921  0.008579
1929-03-28  -0.053577 -0.002647  0.008589  0.005646
1929-04-30   0.004087  0.010386 -0.000616 -0.000598
1929-05-31  -0.105869 -0.054403  0.021996  0.011190
...               ...       ...       ...       ...
1963-02-28   0.020659 -0.023603 -0.000915  0.011828
1963-03-29   0.004649 -0.003947  0.005240  0.005980
1963-04-30  -0.004995  0.017875  0.006440  0.009104
1963-05-31   0.023981  0.021130 -0.000887  0.021555
1963-06-28   0.004958 -0.012085 -0.001562 -0.002025

[414 rows x 4 columns]


In [10]:
def fm_summary(p):
    s = p.describe().T
    s['std_error'] = s['std']/np.sqrt(s['count'])
    s['tstat'] = s['mean']/s['std_error']
    return s[['mean','std_error','tstat']]

print(fm_summary(gamma))


               mean  std_error     tstat
Intercept  0.019000   0.003188  5.960420
beta       0.001663   0.002495  0.666646
ln_melag  -0.002957   0.000581 -5.090217
ln_bmlag   0.002308   0.000849  2.719111


## 2: Can you reject the CAPM based off the results?



Yes we can. First off, beta is statistically insignificant since the tstat is below 1.96.
secondly, ln_melag is nonzero and is very significant statistically with a tstat of 5.09
thirdly, ln_bmlag is nonzero and is also very significant statistically with a tstat of 2.72

These results imply that there is very little correlation between the risk of the security and the return of that security. Since ln_melag is negative that infers smaller stocks make more than the CAPM predicts and vice versa. Ln_bmlag being positive shows that stocks with a higher book to market value ratio outperform stocks that have a lower ratio.

## 3: Estimate the fama mcbeth in a new way:

In [13]:
# this time we only use beta
lm = 'ret ~ 1 + beta'
gamma1 = df.groupby('caldt').apply(ols_coef,lm,include_groups=False)
print(gamma1)

            Intercept      beta
caldt                          
1929-01-31   0.043348 -0.008725
1929-02-28  -0.008727  0.016436
1929-03-28  -0.023803 -0.002481
1929-04-30   0.002069  0.010296
1929-05-31  -0.031596 -0.052384
...               ...       ...
1963-02-28   0.014295 -0.026479
1963-03-29   0.029344 -0.007619
1963-04-30   0.025143  0.012851
1963-05-31   0.016945  0.014617
1963-06-28  -0.002358 -0.011021

[414 rows x 2 columns]


In [14]:
print(fm_summary(gamma1))

              mean  std_error     tstat
Intercept  0.00767   0.001921  3.992866
beta       0.00522   0.002853  1.829372


### 4. Based on your results in question (4), can you reject the CAPM? Explain. Is it even possible to ever reject the CAPM with a regression specification like the one you used in question (4)?

Yes we can reject the CAPM since the beta is insignificant with a Tstat of 1.82 which is below the general rule of tstat > 1.96 to be statistically significant. Since the Beta is insignificant, investors are not rewarded for market risk in their portfolio. 

We cannot fully reject the CAPM with a regression spec like the one I used previously since you need to worry about the joint hypothesis problem. The market could be inefficient and the CAPM could still be good or the contrary could be true.